In [7]:
# --- Đoạn 1: kết nối tới BigQuery ---
from google.cloud import bigquery          # nạp thư viện BigQuery của Google

PROJECT = "bq-learning-510104"             # tên project, lưu vào biến để dùng lại
client = bigquery.Client(project=PROJECT, location="US")
# client = "tay cầm" để Python nói chuyện với BigQuery, mọi lệnh sau đều gọi qua nó
# location="US": dữ liệu của bạn nằm ở US, job phải chạy cùng vùng nếu không sẽ báo không thấy bảng


# --- Đoạn 2: hàm check() = xem trước chi phí, KHÔNG chạy query thật ---
def check(sql):
    """Dry run: ước lượng số byte sẽ quét, KHÔNG chạy query."""
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    # dry_run=True: BigQuery chỉ kiểm tra câu SQL và đếm byte sẽ quét, không chạy thật (không tốn gì)
    # use_query_cache=False: tắt bộ nhớ đệm, để lần đo nào cũng tính như quét thật
    job = client.query(sql, job_config=cfg)             # gửi câu SQL kèm cấu hình trên
    mb = job.total_bytes_processed / 1024**2            # đổi byte sang MB
    print(f"Ước tính quét: {mb:.2f} MB")                # :.2f = làm tròn 2 chữ số thập phân
    return job.total_bytes_processed                    # trả số byte về để lưu và so sánh sau


# --- Đoạn 3: hàm run() = chạy thật, có "cầu chì" ---
def run(sql, max_mb=100):
    """Chạy thật, có cầu chì: vượt max_mb thì query bị từ chối."""
    cfg = bigquery.QueryJobConfig(
        maximum_bytes_billed=int(max_mb * 1024**2), use_query_cache=False
    )
    # maximum_bytes_billed = cầu chì: nếu query định tính tiền quá max_mb MB thì BigQuery
    # từ chối chạy (không mất gì). int(...) đổi MB ra số byte nguyên
    job = client.query(sql, job_config=cfg)
    rows = list(job.result())          # job.result() = chờ query xong; list(...) gom các dòng kết quả
    print(f"processed {job.total_bytes_processed/1024**2:.2f} MB | billed {job.total_bytes_billed/1024**2:.2f} MB")
    # processed = số byte thực tế được quét
    # billed    = số byte bị tính tiền (làm tròn lên, tối thiểu 10 MB mỗi bảng nên bảng nhỏ luôn ra ~10 MB)
    return rows


# --- Đoạn 4: tạo dataset lab để chứa bảng thí nghiệm ---
ds = bigquery.Dataset(f"{PROJECT}.lab")    # khai báo dataset tên bq-learning-510104.lab
ds.location = "US"                         # cùng vùng US với dwh (bắt buộc để copy dữ liệu qua lại)
client.create_dataset(ds, exists_ok=True)  # exists_ok=True: đã có thì bỏ qua, không báo lỗi
print("OK, dataset lab đã sẵn sàng")

OK, dataset lab đã sẵn sàng


In [8]:
for t in client.list_tables(f"{PROJECT}.dwh"):     # đi qua từng bảng trong dataset dwh
    tb = client.get_table(t)                        # lấy thông tin chi tiết của bảng đó
    print(f"\n{tb.table_id}  ({tb.num_rows} dòng, {tb.num_bytes/1024**2:.1f} MB)")
    # in tên bảng, số dòng, dung lượng (MB); \n để xuống dòng cho dễ đọc
    print([(s.name, s.field_type) for s in tb.schema])
    # in danh sách cột dạng (tên cột, kiểu dữ liệu), giống tab Schema trên web


dim_date  (3287 dòng, 0.2 MB)
[('date_key', 'INTEGER'), ('full_date', 'DATE'), ('year', 'INTEGER'), ('quarter', 'INTEGER'), ('month', 'INTEGER'), ('day_name', 'STRING')]

dim_products  (29120 dòng, 2.5 MB)
[('product_id', 'INTEGER'), ('name', 'STRING'), ('category', 'STRING'), ('brand', 'STRING'), ('department', 'STRING')]

dim_users  (100000 dòng, 4.6 MB)
[('user_id', 'INTEGER'), ('gender', 'STRING'), ('age', 'INTEGER'), ('state', 'STRING'), ('country', 'STRING'), ('traffic_source', 'STRING')]

fact_order_items  (180771 dòng, 12.8 MB)
[('order_item_id', 'INTEGER'), ('order_id', 'INTEGER'), ('user_id', 'INTEGER'), ('product_id', 'INTEGER'), ('order_date_key', 'INTEGER'), ('status', 'STRING'), ('sale_price', 'FLOAT'), ('cost', 'FLOAT'), ('gross_profit', 'FLOAT')]


In [9]:
# Lấy ngày nhỏ nhất, ngày lớn nhất, tổng số dòng, và số dòng không nối được với dim_date
sql = f"""
SELECT MIN(d.full_date) AS min_d,          -- ngày sớm nhất của đơn hàng
       MAX(d.full_date) AS max_d,          -- ngày muộn nhất
       COUNT(*)         AS n,              -- tổng số dòng của fact
       COUNTIF(d.full_date IS NULL) AS n_null   -- số dòng không tìm được ngày tương ứng
FROM `{PROJECT}.dwh.fact_order_items` f
LEFT JOIN `{PROJECT}.dwh.dim_date` d       -- LEFT JOIN: giữ mọi dòng của fact, kể cả dòng không khớp
  ON f.order_date_key = d.date_key         -- nối khóa số của fact với khóa số của dim_date
"""
check(sql)            # xem trước số byte sẽ quét
rows = run(sql)       # chạy thật, có cầu chì 100 MB
r = rows[0]           # kết quả chỉ có 1 dòng
print(r.min_d, r.max_d, r.n, r.n_null)

Ước tính quét: 1.43 MB
processed 1.43 MB | billed 20.00 MB
2019-01-11 2026-10-04 180771 0


In [10]:
# Liệt kê mọi bảng trong dataset lab để biết bảng nào đang có thật
for t in client.list_tables(f"{PROJECT}.lab"):
    tb = client.get_table(t)                      # lấy thông tin chi tiết của bảng
    print(tb.table_id, tb.num_rows, tb.expires)   # tên bảng, số dòng, thời điểm bảng tự hết hạn (None = không hết hạn)

fact_items_part 22817 None
fact_items_plain 180771 2026-12-04 02:24:27.547000+00:00


In [11]:
# Phần SELECT dùng chung cho cả hai bảng: toàn bộ cột của fact + thêm cột order_date (kiểu DATE)
base = f"""
SELECT f.*, d.full_date AS order_date      -- f.* = mọi cột cũ; order_date = cột ngày mới
FROM `{PROJECT}.dwh.fact_order_items` f
LEFT JOIN `{PROJECT}.dwh.dim_date` d
  ON f.order_date_key = d.date_key
"""

# Bảng 1: bản thường, KHÔNG partition (dùng để so sánh)
# CREATE OR REPLACE TABLE: nếu bảng đã có thì ghi đè (chỉ dùng tên bảng trong lab!)
run(f"CREATE OR REPLACE TABLE `{PROJECT}.lab.fact_items_plain` AS {base}")

# Bảng 2: giống hệt bảng 1 nhưng chia ngăn theo THÁNG của order_date
# DATE_TRUNC(order_date, MONTH): cắt ngày về ngày đầu tháng, nên mỗi tháng là một ngăn
run(f"""
CREATE OR REPLACE TABLE `{PROJECT}.lab.fact_items_part`
PARTITION BY DATE_TRUNC(order_date, MONTH) AS {base}
""")

processed 12.85 MB | billed 20.00 MB
processed 12.85 MB | billed 20.00 MB


[]

In [13]:
for name in ["fact_items_recent", "fact_items_part"]:
    tb = client.get_table(f"{PROJECT}.lab.{name}")
    print(name, tb.num_rows, f"{tb.num_bytes/1024**2:.1f} MB", tb.time_partitioning is not None)
    # cột cuối: True nếu bảng có partition, False nếu không

NotFound: 404 GET https://bigquery.googleapis.com/bigquery/v2/projects/bq-learning-510104/datasets/lab/tables/fact_items_recent?prettyPrint=false: Not found: Table bq-learning-510104:lab.fact_items_recent

In [14]:
sql = f"""
SELECT DATE_TRUNC(order_date, MONTH) AS thang,   -- gom theo tháng
       COUNT(*) AS n                              -- số dòng mỗi tháng
FROM `{PROJECT}.lab.fact_items_part`
GROUP BY thang
ORDER BY thang
"""
for r in run(sql):            # run() trả về danh sách dòng kết quả
    print(r.thang, r.n)

processed 0.17 MB | billed 10.00 MB
2026-08-01 8178
2026-09-01 13393
2026-10-01 1246


In [15]:
for ds_name in ["raw", "dwh", "lab"]:
    for t in client.list_tables(f"{PROJECT}.{ds_name}"):
        tb = client.get_table(t)               # lấy thông tin chi tiết bảng
        print(ds_name, tb.table_id, tb.expires)  # expires = thời điểm bảng tự bị xóa (None = không hết hạn)

raw us_states 2026-12-02 05:54:54.415000+00:00
raw weather_csv_py 2026-12-02 05:52:22.768000+00:00
raw weather_csv_ui 2026-12-02 05:25:01.582000+00:00
raw weather_daily 2026-11-30 11:03:43.775000+00:00
raw weather_parquet_py 2026-12-02 05:47:16.400000+00:00
raw weather_parquet_ui 2026-12-02 05:30:10.180000+00:00
dwh dim_date 2026-11-29 09:17:01.115000+00:00
dwh dim_products 2026-11-29 09:40:20.682000+00:00
dwh dim_users 2026-11-29 09:30:15.305000+00:00
dwh fact_order_items 2026-11-29 10:07:36.195000+00:00
lab fact_items_part None
lab fact_items_plain 2026-12-04 02:25:33.622000+00:00


In [16]:
_ = run(f"""
CREATE OR REPLACE TABLE `{PROJECT}.lab.fact_items_recent` AS   -- tên MỚI, không đè fact_items_plain
SELECT * FROM `{PROJECT}.lab.fact_items_part`                  -- lấy y nguyên các dòng đang có trong bảng partition
""")

processed 1.79 MB | billed 10.00 MB


In [17]:
for name in ["fact_items_recent", "fact_items_part"]:
    tb = client.get_table(f"{PROJECT}.lab.{name}")
    print(name, tb.num_rows, f"{tb.num_bytes/1024**2:.1f} MB", tb.time_partitioning is not None)
    # cột cuối: True nếu bảng có partition, False nếu không

fact_items_recent 22817 1.8 MB False
fact_items_part 22817 1.8 MB True


In [18]:
# Tên đầy đủ của hai bảng cần so sánh (có dấu ` bao quanh để dùng trong SQL)
plain = f"`{PROJECT}.lab.fact_items_recent`"   # bảng thường, 22817 dòng
part  = f"`{PROJECT}.lab.fact_items_part`"     # bảng partition theo tháng, cùng 22817 dòng

# Liệt kê các tháng còn trong bảng partition, kèm số dòng mỗi tháng
rows = run(f"""
SELECT DATE_TRUNC(order_date, MONTH) AS thang,   -- cắt ngày về đầu tháng để gom theo tháng
       COUNT(*) AS n                              -- số dòng mỗi tháng
FROM {part}
GROUP BY thang
ORDER BY thang
""")
for r in rows:
    print(r.thang, r.n)

# Chọn tháng có NHIỀU dòng nhất để thử (tránh tháng dở dang như tháng hiện tại)
# max(..., key=lambda r: r.n): tìm phần tử có r.n lớn nhất; lambda là hàm nhỏ viết tắt, ở đây chỉ nói "so theo n"
thang = max(rows, key=lambda r: r.n).thang
print("Tháng dùng để thử:", thang)

processed 0.17 MB | billed 10.00 MB
2026-08-01 8178
2026-09-01 13393
2026-10-01 1246
Tháng dùng để thử: 2026-09-01


In [19]:
# Hàm đo bằng dry run, trả về số KB ước tính sẽ quét (dùng KB vì bảng nhỏ, MB làm tròn sẽ ra toàn 0.00)
def kb(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)   # dry_run: chỉ đếm, không chạy thật
    return client.query(sql, job_config=cfg).total_bytes_processed / 1024

# Bốn câu query mẫu; {t} sẽ được thay bằng tên bảng, {m} bằng tháng đã chọn
tests = {
    "1. Lọc theo THÁNG (cột partition)":
        "SELECT SUM(sale_price) FROM {t} "
        "WHERE order_date >= DATE '{m}' AND order_date < DATE_ADD(DATE '{m}', INTERVAL 1 MONTH)",
    "2. Không lọc gì":
        "SELECT SUM(sale_price) FROM {t}",
    "3. Lọc theo cột KHÁC (sale_price)":
        "SELECT SUM(sale_price) FROM {t} WHERE sale_price > 50",
    "4. SELECT * ... LIMIT 10":
        "SELECT * FROM {t} LIMIT 10",
}

# Chạy từng câu trên bảng thường và bảng partition rồi in cạnh nhau
for name, tpl in tests.items():
    a = kb(tpl.format(t=plain, m=thang))   # tpl.format: thay {t} và {m} vào câu SQL mẫu
    b = kb(tpl.format(t=part,  m=thang))
    print(f"{name:38s} thường: {a:9.1f} KB | partition: {b:9.1f} KB")

1. Lọc theo THÁNG (cột partition)      thường:     356.5 KB | partition:     209.3 KB
2. Không lọc gì                        thường:     178.3 KB | partition:     178.3 KB
3. Lọc theo cột KHÁC (sale_price)      thường:     178.3 KB | partition:     178.3 KB
4. SELECT * ... LIMIT 10               thường:    1832.7 KB | partition:    1832.7 KB


In [20]:
# ALTER TABLE ... SET OPTIONS: đổi cài đặt của bảng đã có, không tạo lại bảng, không đụng dữ liệu
# require_partition_filter = TRUE: từ giờ mọi query vào bảng này PHẢI có điều kiện lọc trên cột partition (order_date)
# Muốn tắt lại: chạy đúng lệnh này nhưng đổi TRUE thành FALSE
# Dùng client.query(...).result() trực tiếp (không dùng run()) vì lệnh ALTER không quét dữ liệu
client.query(f"ALTER TABLE {part} SET OPTIONS (require_partition_filter = TRUE)").result()

tb = client.get_table(f"{PROJECT}.lab.fact_items_part")   # đọc lại thông tin bảng để kiểm tra
print("require_partition_filter =", tb.require_partition_filter)

require_partition_filter = True


In [21]:
# Câu này không có WHERE trên order_date, nên ổ khóa phải chặn nó
try:
    client.query(f"SELECT SUM(sale_price) FROM {part}").result()
    print("Không báo lỗi, bất ngờ, gửi mình xem")      # nếu chạy được nghĩa là ổ khóa chưa hoạt động
except Exception as e:
    print(type(e).__name__)           # tên loại lỗi
    print(str(e)[:300])               # 300 ký tự đầu của thông báo lỗi (đủ để đọc lý do)

BadRequest
400 Cannot query over table 'bq-learning-510104.lab.fact_items_part' without a filter over column(s) 'order_date' that can be used for partition elimination; reason: invalidQuery, location: query, message: Cannot query over table 'bq-learning-510104.lab.fact_items_part' without a filter over column(


In [22]:
# Câu này có điều kiện trên order_date nên qua được ổ khóa
rows = run(f"""
SELECT SUM(sale_price) AS tong
FROM {part}
WHERE order_date >= DATE '{thang}'
  AND order_date <  DATE_ADD(DATE '{thang}', INTERVAL 1 MONTH)
""")
print(rows[0].tong)

processed 0.20 MB | billed 10.00 MB
793533.3008778095


In [23]:
# kb(sql): hàm đo bằng dry run ở Cell 11, trả về số KB ước tính sẽ quét

# Thử 1: SELECT * ... LIMIT 10 KHÔNG có WHERE trên order_date, khi ổ khóa đang BẬT
try:
    print("Thử 1:", kb(f"SELECT * FROM {part} LIMIT 10"), "KB")
except Exception as e:
    print("Thử 1:", type(e).__name__, "->", str(e)[:120])   # in tên lỗi và 120 ký tự đầu của thông báo

# Thử 2: cùng câu, nhưng có lọc theo tháng trên cột partition
print("Thử 2:", kb(f"""
SELECT * FROM {part}
WHERE order_date >= DATE '{thang}'
  AND order_date <  DATE_ADD(DATE '{thang}', INTERVAL 1 MONTH)
LIMIT 10
"""), "KB")

Thử 1: BadRequest -> 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/bq-learning-510104/jobs?prettyPrint=false: Cannot query ov
Thử 2: 1075.8701171875 KB


In [24]:
# Ba câu query trên cùng một bảng thường, chỉ khác số cột được chọn
tests = {
    "1 cột (sale_price)":        f"SELECT sale_price FROM {plain}",
    "2 cột (sale_price, status)": f"SELECT sale_price, status FROM {plain}",
    "mọi cột (SELECT *)":         f"SELECT * FROM {plain}",
}

# kb(sql): dry run, trả về số KB ước tính sẽ quét (định nghĩa ở Cell 11)
for name, sql in tests.items():
    print(f"{name:28s} {kb(sql):9.1f} KB")

1 cột (sale_price)               178.3 KB
2 cột (sale_price, status)       406.7 KB
mọi cột (SELECT *)              1832.7 KB


In [25]:
# list_rows: đọc thẳng vài dòng đầu của bảng, KHÔNG tạo query job nên không tính byte quét
# max_results=5: chỉ lấy 5 dòng
rows = client.list_rows(f"{PROJECT}.dwh.fact_order_items", max_results=5)

for r in rows:
    print(dict(r))       # dict(r): đổi mỗi dòng thành dạng {tên_cột: giá_trị} cho dễ đọc

{'order_item_id': 95881, 'order_id': 66166, 'user_id': 52883, 'product_id': 14235, 'order_date_key': 20220307, 'status': 'Returned', 'sale_price': 0.0199999995529651, 'cost': 0.0082999997779726, 'gross_profit': 0.011699999774992502}
{'order_item_id': 146191, 'order_id': 100843, 'user_id': 80899, 'product_id': 14235, 'order_date_key': 20230503, 'status': 'Processing', 'sale_price': 0.0199999995529651, 'cost': 0.0082999997779726, 'gross_profit': 0.011699999774992502}
{'order_item_id': 56441, 'order_id': 38993, 'user_id': 31029, 'product_id': 14235, 'order_date_key': 20241106, 'status': 'Complete', 'sale_price': 0.0199999995529651, 'cost': 0.0082999997779726, 'gross_profit': 0.011699999774992502}
{'order_item_id': 144724, 'order_id': 99821, 'user_id': 80048, 'product_id': 14235, 'order_date_key': 20200623, 'status': 'Returned', 'sale_price': 0.0199999995529651, 'cost': 0.0082999997779726, 'gross_profit': 0.011699999774992502}
{'order_item_id': 98776, 'order_id': 68141, 'user_id': 54485, '

In [26]:
import time    # thư viện đo thời gian

# Hai bảng cần đọc: dim_date rất nhỏ (0,2 MB), fact_order_items lớn hơn (12,8 MB)
for name in ["dim_date", "fact_order_items"]:
    t0 = time.time()                                          # ghi lại giờ bắt đầu
    rows = list(client.list_rows(f"{PROJECT}.dwh.{name}"))    # không có max_results nên đọc HẾT bảng; list(...) gom hết vào bộ nhớ
    giay = time.time() - t0                                   # thời gian đã trôi qua, tính bằng giây
    print(f"{name:18s} {len(rows):>8} dòng | {giay:.1f} giây")
    # len(rows) = số dòng đã đọc, phải khớp với số dòng đã thấy ở Cell 2 (3287 và 180771)

dim_date               3287 dòng | 1.7 giây
fact_order_items     180771 dòng | 40.0 giây
